## Adapter

---

An **interface** is a finite set of typed method signatures. For a method $g$ with parameter type $P$ and return type $R$ we write $g : P \rightarrow R$.

Let:

- $I_1$ = the **adaptee** interface — the signatures the old system exposes. It contains a method $f : P_1 \rightarrow R_1$.
- $I_2$ = the **target** interface — the signatures the client expects to call. It contains a method $g : P_2 \rightarrow R_2$.

In general $P_2 \neq P_1$ and $R_2 \neq R_1$: the shapes do not match, so the client cannot call $f$ directly. Adaptation requires two **pure translation functions**, one per direction of data flow:

$$t : P_2 \rightarrow P_1 \qquad \text{(arguments: client format} \rightarrow \text{adaptee format)}$$

$$r : R_1 \rightarrow R_2 \qquad \text{(result: adaptee format} \rightarrow \text{client format)}$$

The **adapter** $A$ is an object that *implements* $I_2$ while holding a reference to an $I_1$. Its realization of $g$ is the composition:

$$\boxed{\,A.g \;=\; r \circ f \circ t\,}$$

Pointwise, for a client request $x \in P_2$:

$$A.g(x) \;=\; r\big(f(t(x))\big)$$

**Type check** — the composition chains, so the adapter is type-indistinguishable from a native $I_2$ method:

$$\underbrace{x}_{P_2} \;\xrightarrow{\;t\;}\; \underbrace{t(x)}_{P_1} \;\xrightarrow{\;f\;}\; \underbrace{f(t(x))}_{R_1} \;\xrightarrow{\;r\;}\; \underbrace{r(f(t(x)))}_{R_2} \qquad\Rightarrow\qquad A.g : P_2 \rightarrow R_2 \;=\; g \in I_2 \;\checkmark$$

As a map on requests, data enters in $I_2$ and exits in $I_1$:

$$A : I_2 \rightarrow I_1, \qquad \text{client} \xrightarrow{\,I_2\,} A \xrightarrow{\,I_1\,} \text{adaptee}$$

**Conditions**

1. **Isolation** — only $A$ references both sides: the client mentions $I_2$ only, the adaptee mentions $I_1$ only.  
   $$\text{client} \perp I_1, \qquad \text{adaptee} \perp I_2$$
2. **No mutation** — $f$, the adaptee, and the signature of $g$ are all unchanged; the adapter adds behavior purely by composition.
3. **Degenerate case** — if the interfaces already align, $t = \mathrm{id}_{P}$ and $r = \mathrm{id}_{R}$, so $A.g = f$: no adapter is needed. The adapter's content is exactly the non-identity part of $t$ and $r$.


### Exercise 1 — Legacy Printer Adapter

---

**Scenario:** You have a modern app that calls `print_document(text)` (the target interface $I_2$). An old legacy printer only understands `old_print(data)` (the adaptee interface $I_1$). You cannot change either side.

**Your task:** Write a `PrinterAdapter` (the adapter $A$) that wraps the legacy printer so the modern app works unchanged.

```python
adapter = PrinterAdapter(LegacyPrinter())   # A wraps I_1
adapter.print_document("Hello")             # client uses I_2
```

**Hints**

- $A$ holds a reference to the adaptee: `self.legacy = LegacyPrinter()`. Then `print_document(text)` calls `self.legacy.old_print(text)` — the translation $g \mapsto f$.
- No inheritance needed. The adapter **has** the legacy printer (composition), it does not **extend** it.


In [1]:
#--------------------------------
# Adaptee (I_1) — you cannot change this

class LegacyPrinter:
    def old_print(self, data):
        return f"[LEGACY] {data}"

#--------------------------------
# Adapter (A) — your task: implement print_document (I_2)

class PrinterAdapter:
    def __init__(self, legacy):
        self.legacy = legacy            # A holds the adaptee

    def print_document(self, text):
        return self.legacy.old_print(text) # call self.legacy.old_print(...)  (g -> f)

#--------------------------------
adapter = PrinterAdapter(LegacyPrinter())   # A wraps I_1
print(adapter.print_document("Hello"))      # client uses I_2  ->  [LEGACY] Hello


[LEGACY] Hello


### Exercise 2 — Third-Party Payment Adapter

---

**Scenario:** Your checkout calls `pay(amount, currency)` (the target interface $I_2$). A third-party SDK exposes `make_payment(total, currency_code)` (the adaptee interface $I_1$). You cannot change the SDK.

**Your task:** Write a `PaymentAdapter` that maps $I_2 \rightarrow I_1$, **including renaming the arguments** — this is the argument-translation step $t$ (the $f^{-1}(x)$ part).

```python
processor = PaymentAdapter(ThirdPartySDK())
checkout(processor)        # works exactly as with a native I_2 processor
```

**Hints**

- Argument renaming is the $t$ transformation: `amount` $\rightarrow$ `total`, `currency` $\rightarrow$ `currency_code`. So `pay(amount, currency)` calls `make_payment(total=amount, currency_code=currency)`.
- Write a `checkout(processor)` function that **only** calls `processor.pay()`. It should work with both a real processor and the adapter — demonstrating **transparency**: the client cannot tell the difference.


In [2]:
#--------------------------------
# Adaptee (I_1) — third-party SDK, you cannot change this

class ThirdPartySDK:
    def make_payment(self, total, currency_code):
        return f"[SDK] charged {total} {currency_code}"

#--------------------------------
# Adapter (A) — your task: implement pay (I_2), renaming args  (t / f^-1)

class PaymentAdapter:
    def __init__(self, sdk):
        self.sdk = sdk                  # A holds the adaptee

    def _currency_code(self, currency): # Logic for code will go here.
        return currency

    def pay(self, amount, currency):
        return self.sdk.make_payment(total=amount, currency_code=self._currency_code(currency))
     # call self.sdk.make_payment(total=..., currency_code=...)

#--------------------------------
# Client (I_2) — only knows .pay(); blind to which interface is underneath

processor = PaymentAdapter(ThirdPartySDK())   # A wraps I_1
print(processor.pay(100, "USD"))                 # -> [SDK] charged 100 USD


[SDK] charged 100 USD
